# Part B - Threads


**Part B — Threads**

13. [Process vs Thread — Concepts](#s13)
14. [Creating and Starting a Thread](#s14)
15. [Multiple Threads — Speed Comparison](#s15)
16. [Daemon Threads](#s16)
17. [Race Conditions](#s17)
18. [threading.Lock — The Fix](#s18)
19. [Semaphore and Event](#s19)
20. [Thread Life Cycle](#s20)
21. [Thread as an OOP Subclass](#s21)

<a id='s13'></a>
## Section 13: Process vs Thread - Concepts

**The Bank Branch Analogy:**

| With 1 cashier | With 5 cashiers |
|---|---|
| Sequential - one customer at a time | Concurrent - 5 customers at once |
| **Process with 1 thread** | **Process with many threads** |

**Feature, Process, and Thread**
| Feature | Process | Thread |
|---|---|---|
| Memory | Private - completely separate | Shared - all threads see the same variables |
| Communication | Slow (files, pipes, sockets) | Fast (shared variables) |
| Startup cost | High | Low |
| Use case | CPU-heavy tasks | I/O-heavy tasks |
| Python module | `multiprocessing` | `threading` |

**The Global Interpreter Lock (GIL):**  
Python's GIL ensures only ONE thread executes Python bytecode at a time.  

- **I/O-bound tasks** (downloading, DB queries, waiting) → threads HELP ✅  
- **CPU-bound tasks** (heavy calculations, image processing) → use `multiprocessing` instead ❌

> **Why threads help for I/O:**  
> While one thread WAITS (for network/disk), another thread can RUN.  
> The total time = the longest single task (not the sum of all tasks).

In [ ]:
import threading
import time

# Demonstrating the speed benefit of threads for I/O-bound tasks
def simulate_download(name, duration):
    """Simulate a file download (I/O wait)."""
    time.sleep(duration)    # represents waiting for data
    print(f"  [{name}] download complete ({duration}s)")

downloads = [("FileA", 2.0), ("FileB", 1.0), ("FileC", 1.5)]

# --- Sequential ---
print("=== Sequential ===")
start = time.perf_counter()
for name, dur in downloads:
    simulate_download(name, dur)
seq_time = time.perf_counter() - start
print(f"Sequential total: {seq_time:.1f}s\n")

# --- Threaded ---
print("=== Threaded ===")
start = time.perf_counter()
threads = []
for name, dur in downloads:
    t = threading.Thread(target=simulate_download, args=(name, dur))
    threads.append(t)
    t.start()      # start ALL threads first
for t in threads:
    t.join()       # then wait for ALL
thr_time = time.perf_counter() - start
print(f"Threaded total:   {thr_time:.1f}s")
print(f"Speed-up: {seq_time / thr_time:.1f}x")

<a id='s14'></a>
## Section 14: Creating and Starting a Thread

```python
import threading

t = threading.Thread(target=my_function, args=(arg1, arg2))
t.start()   # begin execution
t.join()    # wait for it to finish
```

**Three-step pattern: Create → Start → Join**

| Step | Call | Purpose |
|---|---|---|
| 1 | `threading.Thread(target=..., args=...)` | Create the Thread object (does NOT start yet) |
| 2 | `t.start()` | Start the thread (begins running in the background) |
| 3 | `t.join()` | Main thread waits here until the worker thread finishes |

> `t.start()` **does NOT wait** for the thread to finish.  
> If you want to wait, you must explicitly call `t.join()`.

In [ ]:
import threading
import time

def process_book_order(student, book, seconds):
    print(f"  [Order] Processing '{book}' for {student}...")
    time.sleep(seconds)          # simulate library database lookup
    print(f"  [Order] Done: '{book}' is ready for {student}.")

# Step 1: Create the Thread object
t = threading.Thread(
    target=process_book_order,
    args=("Alice", "Fluent Python", 1.5)
)

print("[Main] Before start")

# Step 2: Start the thread
t.start()

# Main thread continues immediately
print("[Main] Thread started - main continues while thread runs")

# Step 3: Wait for thread to complete
t.join()

print("[Main] Thread finished. Program ends.")

# Query thread status
print(f"[Main] Thread is alive: {t.is_alive()}")

<a id='s15'></a>
## Section 15: Multiple Threads - Speed Comparison

**Critical pattern: Start ALL threads first, then join ALL.**

If you start and join inside the same loop, you get **sequential** behaviour:

```python
# WRONG - sequential! (join blocks before next start)
for t in threads:
    t.start()
    t.join()   # ← waits here before starting the next thread

# CORRECT - concurrent
for t in threads: t.start()   # start ALL
for t in threads: t.join()    # then join ALL
```

In [ ]:
import threading
import time

def process_order(student, book, duration):
    time.sleep(duration)
    print(f"  [{student}] '{book}' ready ({duration}s)")

orders = [
    ("Alice",   "Fluent Python",   2.0),
    ("Bob",     "Clean Code",      1.0),
    ("Charlie", "Python Cookbook", 1.5),
    ("Diana",   "Deep Learning",   1.8),
]

# --- WRONG approach: start-join in same loop ---
print("=== Wrong (sequential despite using threads) ===")
threads = []
start = time.perf_counter()
for student, book, dur in orders:
    t = threading.Thread(target=process_order, args=(student, book, dur))
    threads.append(t)
    t.start()
    t.join()    # blocks here!
print(f"Wrong: {time.perf_counter() - start:.1f}s  (sum of all durations)\n")

# --- CORRECT approach: start all, then join all ---
print("=== Correct (truly concurrent) ===")
threads = []
start = time.perf_counter()
for student, book, dur in orders:
    t = threading.Thread(target=process_order, args=(student, book, dur))
    threads.append(t)
    t.start()    # start ALL
for t in threads:
    t.join()     # then join ALL
print(f"Correct: {time.perf_counter() - start:.1f}s  (≈ longest single task)")

<a id='s16'></a>
## Section 16: Daemon Threads

A **daemon thread** is automatically killed when the **main thread** ends -  
even if the daemon thread has not finished its work.

| | `daemon=False` (default) | `daemon=True` |
|---|---|---|
| **Lifetime** | Must finish before program exits | Auto-killed when main thread ends |
| **Use for** | Important work that must complete | Background helpers (logging, autosave) |

> **Analogy:** Daemon threads are like background music in a game.  
> When the game ends, the music stops automatically - you don't need to explicitly stop it.

In [ ]:
import threading
import time

def background_autosave():
    count = 0
    while True:
        count += 1
        time.sleep(1)
        print(f"  [AutoSave] Game state saved (checkpoint #{count})")

# daemon=True - this thread will be killed when main thread ends
save_thread = threading.Thread(
    target=background_autosave,
    daemon=True           # <-- the key flag
)
save_thread.start()

print("[Game] Game started. Playing for 3 seconds...")
time.sleep(3)
print("[Game] Game over.")
# Daemon thread is automatically killed here
# No need to call save_thread.join()
print("[Game] Program exits.")

<a id='s17'></a>
## Section 17: Race Conditions

A **race condition** occurs when two or more threads access and modify shared data  
at the same time, producing incorrect results.

**The bank account problem:**

```
Balance: £1000

Thread 1 reads: £1000      Thread 2 reads: £1000
Thread 1 adds:  + £200     Thread 2 adds:  + £300
Thread 1 writes: £1200     Thread 2 writes: £1300

Final: £1300  ←  WRONG! Should be £1500
```

**The gap** between READ and WRITE is where threads can interleave.

**Symptoms of a race condition:**  
- Results differ on every run  
- Bug disappears when you add `print()` statements  
- Works on your machine but fails in testing

In [ ]:
import threading

# UNSAFE - race condition demonstrated
balance = 0

def deposit(amount, count):
    global balance
    for _ in range(count):
        current = balance          # READ  ← race condition gap here
        balance = current + amount # WRITE

balance = 0
t1 = threading.Thread(target=deposit, args=(1, 100_000))
t2 = threading.Thread(target=deposit, args=(1, 100_000))
t1.start()
t2.start()
t1.join()
t2.join()

print(f"Expected: 200000")
print(f"Got:      {balance}")
if balance != 200_000:
    print("Race condition detected! Some deposits were lost.")
else:
    print("Lucky - got correct result this time. Run again to see the race condition.")

<a id='s18'></a>
## Section 18: `threading.Lock` - The Fix

A **Lock** ensures that only **one thread** can execute the **critical section** at a time.  
All other threads wait until the lock is released.

```python
lock = threading.Lock()

with lock:          # acquire the lock (others wait here)
    # critical section - only one thread at a time
    ...             # lock released automatically when block ends
```

> ✅ **Always use `with lock:`** - it guarantees the lock is released even if an exception occurs.  
> ❌ **Never** call `lock.acquire()` / `lock.release()` manually - you might forget to release.

In [ ]:
import threading

# SAFE - with lock
safe_balance = 0
lock = threading.Lock()

def safe_deposit(amount, count):
    global safe_balance
    for _ in range(count):
        with lock:                         # only ONE thread at a time
            current = safe_balance
            safe_balance = current + amount

safe_balance = 0
t1 = threading.Thread(target=safe_deposit, args=(1, 100_000))
t2 = threading.Thread(target=safe_deposit, args=(1, 100_000))
t1.start()
t2.start()
t1.join()
t2.join()

print(f"Expected: 200000")
print(f"Got:      {safe_balance}")
print("Result is always correct with a lock!")

# Demonstrate: lock blocks other threads
import time

access_lock = threading.Lock()

def access_resource(name):
    print(f"  {name}: waiting for lock...")
    with access_lock:
        print(f"  {name}: inside critical section")
        time.sleep(0.5)
        print(f"  {name}: leaving critical section")

print("\n--- Lock blocking demonstration ---")
threads = [threading.Thread(target=access_resource, args=(f"Thread-{i}",))
           for i in range(3)]
for t in threads: t.start()
for t in threads: t.join()

<a id='s19'></a>
## Section 19: `Semaphore` and `Event`

**`threading.Semaphore(n)` - limit N concurrent threads:**  
Like a bouncer at a venue: maximum N people allowed inside at once.

```python
sem = threading.Semaphore(3)  # max 3 threads at a time
with sem:
    ...  # at most 3 threads in this block simultaneously
```

**`threading.Event()` - signal between threads:**  
Like a starting gun at a race - one shot signals all runners to go.

```python
event = threading.Event()
event.set()       # signal: is_set() becomes True
event.clear()     # reset: is_set() becomes False
event.wait()      # block until set() is called
event.is_set()    # check without blocking
```

| Tool | Purpose | Max threads |
|---|---|---|
| `Lock` | Mutual exclusion | 1 |
| `Semaphore(n)` | Limit concurrent access | n |
| `Event` | One-to-many signal | - |

In [ ]:
import threading
import time

# Semaphore - computer lab with only 2 seats
lab = threading.Semaphore(2)  # max 2 students at once

def use_lab(student):
    print(f"  {student}: waiting for a seat...")
    with lab:     # blocks if 2 students already inside
        print(f"  {student}: seated and working")
        time.sleep(1.0)   # work in lab
        print(f"  {student}: finished and leaving")

print("=== Semaphore demo ===")
students = ["Alice", "Bob", "Charlie", "Diana", "Eve"]
threads  = [threading.Thread(target=use_lab, args=(s,)) for s in students]
for t in threads: t.start()
for t in threads: t.join()

print()

# Event - game over signal
print("=== Event demo ===")
game_over = threading.Event()

def score_tracker():
    score = 0
    while not game_over.is_set():
        score += 10
        time.sleep(0.3)
    print(f"  [Score Tracker] Game over. Final score: {score}")

t = threading.Thread(target=score_tracker, daemon=True)
t.start()

print("  [Game] Playing...")
time.sleep(1.5)
print("  [Game] Triggering game over!")
game_over.set()   # signal the score tracker to stop
t.join()

<a id='s20'></a>
## Section 20: Thread Life Cycle

A thread moves through five states during its lifetime:

```
  ┌──────────┐  .start()  ┌──────────┐  CPU granted  ┌──────────┐
  │   NEW    │ ──────────▶│ RUNNABLE │ ─────────────▶│ RUNNING  │
  └──────────┘            └──────────┘               └────┬─────┘
                                                           │
                  ┌────────────────────────────────────────┤
                  │  lock, sleep(), join(), I/O wait        │
                  ▼                                        │
           ┌──────────────┐   wait over                   │
           │  BLOCKED /   │ ◀─────────────────────────────┘
           │  WAITING     │
           └──────┬───────┘
                  │ condition met
                  ▼
           ┌────────────┐
           │ TERMINATED │  run() returned or unhandled exception
           └────────────┘
```

| State | Description |
|---|---|
| NEW | Thread created, `start()` not yet called |
| RUNNABLE | `start()` called; waiting for CPU |
| RUNNING | OS gave CPU; `run()` is executing |
| BLOCKED / WAITING | Paused for lock, sleep, join, or I/O |
| TERMINATED | `run()` returned or exception raised |

> ⚠️ **A thread CANNOT be restarted once terminated.**  
> ⚠️ **Unhandled exceptions inside threads cause silent termination!**  
> Always wrap your thread function body in `try / except`.

In [ ]:
import threading
import time

# Monitoring thread state with is_alive()
def slow_task():
    time.sleep(1.5)

t = threading.Thread(target=slow_task)
print(f"Before start:  is_alive = {t.is_alive()}")
t.start()
print(f"After start:   is_alive = {t.is_alive()}")
time.sleep(0.5)
print(f"Midway (0.5s): is_alive = {t.is_alive()}")
t.join()
print(f"After join:    is_alive = {t.is_alive()}")

print()

# Silent exception demo - thread dies without warning
def buggy_task():
    print("  [BuggyTask] Starting...")
    raise ValueError("Something went wrong inside the thread!")
    print("  [BuggyTask] This never prints.")  # unreachable

def safe_task():
    try:
        buggy_task()
    except Exception as e:
        print(f"  [SafeTask] Caught exception: {e}")

print("--- Without exception handling (exception is silent) ---")
t_bad = threading.Thread(target=buggy_task)
t_bad.start()
t_bad.join()
print("  [Main] Main continues - did you notice the thread crashed?")

print()
print("--- With exception handling ---")
t_good = threading.Thread(target=safe_task)
t_good.start()
t_good.join()
print("  [Main] Exception was caught and handled.")

<a id='s21'></a>
## Section 21: Thread as an OOP Subclass

You can subclass `threading.Thread` when your thread needs to **store state**  
or when you want to keep the thread logic self-contained in a class.

**Rules:**
1. Always call `super().__init__()` in `__init__`
2. Override `run()` - NOT `start()` - with your logic
3. Store results as instance attributes (much cleaner than global variables)

> This connects directly back to **Unit 2 - OOP**:  
> `StudentWorker` IS-A `Thread` (inheritance), HAS-A `result` (attribute).

In [ ]:
import threading
import time

class StudentWorker(threading.Thread):
    """
    A Thread subclass that simulates a student completing an assignment.
    Results are stored as instance attributes for easy retrieval.
    """

    def __init__(self, student_name, subject, time_needed):
        super().__init__()     # MUST call Thread.__init__()
        self.student_name = student_name
        self.subject      = subject
        self.time_needed  = time_needed
        self.result       = None   # will be set in run()
        self.grade        = None

    def run(self):
        # This is called by t.start() - override run(), not start()
        print(f"  [{self.student_name}] Starting {self.subject}...")
        time.sleep(self.time_needed)   # simulate work
        self.grade  = "A" if self.time_needed < 1.5 else "B"
        self.result = f"{self.student_name} - {self.subject}: Grade {self.grade}"
        print(f"  [{self.student_name}] Done!")

# Create and start workers
workers = [
    StudentWorker("Alice",   "Maths",      1.0),
    StudentWorker("Bob",     "English",    0.7),
    StudentWorker("Charlie", "Programming",1.8),
    StudentWorker("Diana",   "Physics",    1.2),
]

for w in workers:
    w.start()

for w in workers:
    w.join()

print("\n=== Results ===")
for w in workers:
    print(f"  {w.result}")